# ReccoBeats Recommendations Pipeline
Fetches ≥5 track recommendations for every row in `spotify_5000.parquet` using the [ReccoBeats API](https://reccobeats.com/docs/apis/reccobeats-api), then exports an enriched parquet `spotify_5000_reccobeats.parquet`.

**API base:** `https://api.reccobeats.com/v1`  
**Key endpoint:** `GET /track/recommendation?seeds=<spotify_track_id>&size=<n>`  
**Rate limits:** Requests are throttled — the pipeline respects 429 responses with exponential back-off.

In [1]:
# ── dependencies ──────────────────────────────────────────────────────────────
# pip install polars requests tqdm

import time
import logging
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests
import polars as pl
from tqdm.auto import tqdm

logging.basicConfig(level=logging.INFO, format="%(levelname)s │ %(message)s")
log = logging.getLogger(__name__)
print(f"Polars {pl.__version__}")

Polars 1.38.1


## Config

In [2]:
# ── config ────────────────────────────────────────────────────────────────────
INPUT_PARQUET  = Path("spotify_5000.parquet")
OUTPUT_PARQUET = Path("spotify_5000_reccobeats.parquet")

API_BASE       = "https://api.reccobeats.com/v1"
RECS_PER_TRACK = 5          # minimum 5 recommendations per seed
MAX_WORKERS    = 8          # concurrent threads — keep low to respect rate limits
RETRY_LIMIT    = 5          # max retries on 429 / 5xx
BACKOFF_BASE   = 2.0        # exponential back-off base (seconds)

# Column in the parquet that holds the Spotify track ID
TRACK_ID_COL   = "track_id"

## 1 · Load source data

In [3]:
df = pl.read_parquet(INPUT_PARQUET)
print(f"Loaded {df.shape[0]:,} rows × {df.shape[1]} cols")
print(f"Columns: {df.columns}")
df.head(3)

Loaded 5,000 rows × 39 cols
Columns: ['track_id', 'original_name', 'original_artist', 'id', 'name', 'title', 'uri', 'type', 'duration', 'duration_ms', 'artists', 'preview_url', 'is_explicit', 'is_playable', 'album', 'release_date', 'metadata_status', 'lyrics', 'lyrics_status', 'stream_count', 'Unnamed: 0', 'artist_name', 'track_name', 'popularity', 'year', 'genre', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_ms_right', 'time_signature']


track_id,original_name,original_artist,id,name,title,uri,type,duration,duration_ms,artists,preview_url,is_explicit,is_playable,album,release_date,metadata_status,lyrics,lyrics_status,stream_count,Unnamed: 0,artist_name,track_name,popularity,year,genre,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms_right,time_signature
str,str,str,str,str,str,str,str,f64,f64,list[struct[3]],str,bool,bool,struct[3],str,str,str,str,f64,i64,str,str,i64,i64,str,f64,f64,i64,f64,i64,f64,f64,f64,f64,f64,f64,i64,i64
"""0yLdNVWF3Srea0uzk55zFn""","""Flowers""","""Miley Cyrus""","""0yLdNVWF3Srea0uzk55zFn""","""Flowers""","""Flowers""","""spotify:track:0yLdNVWF3Srea0uz…","""track""",200454.0,200454.0,"[{""5YGY8feqx7naU7z4HrwZM6"",""Miley Cyrus"",""spotify:artist:5YGY8feqx7naU7z4HrwZM6""}]","""https://p.scdn.co/mp3-preview/…",false,true,"{[{300,""https://image-cdn-fa.spotifycdn.com/image/ab67616d00001e02f429549123dbe8552764ba1d"",300}, {64,""https://image-cdn-fa.spotifycdn.com/image/ab67616d00004851f429549123dbe8552764ba1d"",64}, {640,""https://image-cdn-fa.spotifycdn.com/image/ab67616d0000b273f429549123dbe8552764ba1d"",640}],"""",""album""}","""2023-01-13""","""ok""","""We were good, we were gold Kin…","""ok_lrclib""",2.8096e9,612503,"""Miley Cyrus""","""Flowers""",100,2023,"""pop""",0.707,0.681,0,-4.325,1,0.0668,0.0632,0.000005,0.0322,0.646,117.999,200455,4
"""4nrPB8O7Y7wsOCJdgXkthe""","""Shakira: Bzrp Music Sessions, …","""Bizarrap""","""4nrPB8O7Y7wsOCJdgXkthe""","""Shakira: Bzrp Music Sessions, …","""Shakira: Bzrp Music Sessions, …","""spotify:track:4nrPB8O7Y7wsOCJd…","""track""",214945.0,214945.0,"[{""716NhGYqD1jl2wI1Qkgq36"",""Bizarrap"",""spotify:artist:716NhGYqD1jl2wI1Qkgq36""}, {""0EmeFodog0BfCgMzAIvKQp"",""Shakira"",""spotify:artist:0EmeFodog0BfCgMzAIvKQp""}]","""https://p.scdn.co/mp3-preview/…",false,true,"{[{300,""https://image-cdn-fa.spotifycdn.com/image/ab67616d00001e02fdecba1ff443653b79ebb189"",300}, {64,""https://image-cdn-fa.spotifycdn.com/image/ab67616d00004851fdecba1ff443653b79ebb189"",64}, {640,""https://image-cdn-fa.spotifycdn.com/image/ab67616d0000b273fdecba1ff443653b79ebb189"",640}],"""",""album""}","""2023-01-11""","""ok""","""Pa tipos como tú-ú-ú-ú-ú Oh-oh…","""ok_lrclib""",1.1536e9,605178,"""Bizarrap""","""Shakira: Bzrp Music Sessions, …",96,2023,"""hip-hop""",0.778,0.632,2,-5.6,0,0.0493,0.274,0.0,0.0915,0.498,122.104,218289,4
"""7oDd86yk8itslrA9HRP2ki""","""Die For You - Remix""","""The Weeknd""","""7oDd86yk8itslrA9HRP2ki""","""Die For You - Remix""","""Die For You - Remix""","""spotify:track:7oDd86yk8itslrA9…","""track""",232857.0,232857.0,"[{""1Xyo4u8uXC1ZmMpatF05PJ"",""The Weeknd"",""spotify:artist:1Xyo4u8uXC1ZmMpatF05PJ""}, {""66CXWjxzNUsdJxJ2JdwvnR"",""Ariana Grande"",""spotify:artist:66CXWjxzNUsdJxJ2JdwvnR""}]","""https://p.scdn.co/mp3-preview/…",false,true,"{[{300,""https://image-cdn-fa.spotifycdn.com/image/ab67616d00001e028de12a274f6e1df6634f57ec"",300}, {64,""https://image-cdn-fa.spotifycdn.com/image/ab67616d000048518de12a274f6e1df6634f57ec"",64}, {640,""https://image-cdn-fa.spotifycdn.com/image/ab67616d0000b2738de12a274f6e1df6634f57ec"",640}],"""",""album""}","""2023-02-24""","""ok""","""I'm findin' ways to articulate…","""ok_lrclib""",3.1894e9,612504,"""The Weeknd""","""Die For You - Remix""",95,2023,"""pop""",0.531,0.525,1,-6.5,0,0.0671,0.232,0.0,0.441,0.502,66.9,232857,4


In [4]:
# Confirm the track_id column exists; adjust TRACK_ID_COL above if needed
assert TRACK_ID_COL in df.columns, (
    f"Column '{TRACK_ID_COL}' not found. Available: {df.columns}"
)

track_ids: list[str] = (
    df
    .select(TRACK_ID_COL)
    .drop_nulls()
    .unique()
    [TRACK_ID_COL]
    .to_list()
)
print(f"{len(track_ids):,} unique, non-null track IDs")

5,000 unique, non-null track IDs


## 2 · ReccoBeats API helpers

In [5]:
SESSION = requests.Session()
SESSION.headers.update({"Accept": "application/json"})


def fetch_recommendations(
    seed_id: str,
    size: int = RECS_PER_TRACK,
    retries: int = RETRY_LIMIT,
) -> list[dict]:
    """
    Call GET /v1/track/recommendation for a single seed Spotify track ID.

    Returns a list of recommendation dicts from the API.  Each dict contains
    at minimum:
        id           – ReccoBeats internal UUID
        href         – Spotify URL  (https://open.spotify.com/track/<spotify_id>)
        trackTitle   – track name
        artists      – list of artist objects
        popularity   – int

    On unrecoverable error returns an empty list so the pipeline continues.
    """
    url    = f"{API_BASE}/track/recommendation"
    params = {"seeds": seed_id, "size": size}

    for attempt in range(1, retries + 1):
        try:
            resp = SESSION.get(url, params=params, timeout=15)

            if resp.status_code == 200:
                data = resp.json()
                # The API returns either a list or an object with a 'content' key
                if isinstance(data, list):
                    return data
                return data.get("content", data.get("data", []))

            elif resp.status_code == 429:
                retry_after = float(resp.headers.get("Retry-After", BACKOFF_BASE ** attempt))
                log.warning("429 rate-limit for %s – sleeping %.1fs", seed_id, retry_after)
                time.sleep(retry_after)

            elif resp.status_code == 400:
                log.warning("400 bad request for seed '%s': %s", seed_id, resp.text[:120])
                return []   # non-retryable

            elif resp.status_code == 404:
                log.debug("404 – track '%s' not found in ReccoBeats", seed_id)
                return []

            else:
                wait = BACKOFF_BASE ** attempt
                log.warning("HTTP %d for %s (attempt %d/%d) – retry in %.1fs",
                            resp.status_code, seed_id, attempt, retries, wait)
                time.sleep(wait)

        except requests.RequestException as exc:
            wait = BACKOFF_BASE ** attempt
            log.warning("Request error for %s: %s – retry in %.1fs", seed_id, exc, wait)
            time.sleep(wait)

    log.error("Exhausted retries for seed '%s'", seed_id)
    return []


def extract_spotify_id(href: str | None) -> str | None:
    """Parse Spotify track ID from a Spotify URL."""
    if not href:
        return None
    # https://open.spotify.com/track/<id>
    parts = href.rstrip("/").split("/")
    if len(parts) >= 2 and parts[-2] == "track":
        return parts[-1]
    return None

### 2a · Quick smoke-test (single track)

In [6]:
sample_id = track_ids[0]
print(f"Smoke-test seed: {sample_id}")

sample_recs = fetch_recommendations(sample_id, size=5)
print(f"Returned {len(sample_recs)} recommendations")

if sample_recs:
    import json
    print(json.dumps(sample_recs[0], indent=2))
else:
    print("⚠ No results – check network / track ID validity")

Smoke-test seed: 42tFTth2jcF7iSo0RBjfJF
Returned 4 recommendations
{
  "id": "1a005fcf-4b4b-4c90-ab53-5c630bd9e1b3",
  "trackTitle": "ID",
  "artists": [
    {
      "id": "d5ccd425-aa0f-4af5-b9e6-24bad074b10e",
      "name": "Young Miko",
      "href": "https://open.spotify.com/artist/3qsKSpcV3ncke3hw52JSMB"
    },
    {
      "id": "aafc7d25-d465-4038-b90a-3451eeed673b",
      "name": "Jowell & Randy",
      "href": "https://open.spotify.com/artist/4IMAo2UQchVFyPH24PAjUs"
    }
  ],
  "durationMs": 235525,
  "isrc": "QZXD92300004",
  "ean": null,
  "upc": null,
  "href": "https://open.spotify.com/track/6rGa59Ibh8CnTRxpftnB53",
  "availableCountries": "AR,AU,AT,BE,BO,BR,BG,CA,CL,CO,CR,CY,CZ,DK,DO,DE,EC,EE,SV,FI,FR,GR,GT,HN,HK,HU,IS,IE,IT,LV,LT,LU,MY,MT,MX,NL,NZ,NI,NO,PA,PY,PE,PH,PL,PT,SG,SK,ES,SE,CH,TW,TR,UY,US,GB,AD,LI,MC,ID,JP,TH,VN,RO,IL,ZA,SA,AE,BH,QA,OM,KW,EG,MA,DZ,TN,LB,JO,PS,IN,BY,KZ,MD,UA,AL,BA,HR,ME,MK,RS,SI,KR,BD,PK,LK,GH,KE,NG,TZ,UG,AG,AM,BS,BB,BZ,BT,BW,BF,CV,CW,DM,FJ,GM,GE

## 3 · Batch fetch with concurrency + progress bar

In [7]:
def fetch_one(seed_id: str) -> tuple[str, list[dict]]:
    """Worker: returns (seed_id, recommendations)."""
    return seed_id, fetch_recommendations(seed_id, size=RECS_PER_TRACK)


results: dict[str, list[dict]] = {}

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
    futures = {pool.submit(fetch_one, tid): tid for tid in track_ids}
    with tqdm(total=len(futures), desc="Fetching recommendations", unit="track") as pbar:
        for future in as_completed(futures):
            seed, recs = future.result()
            results[seed] = recs
            pbar.update(1)

# Summary
n_success  = sum(1 for v in results.values() if len(v) >= RECS_PER_TRACK)
n_partial  = sum(1 for v in results.values() if 0 < len(v) < RECS_PER_TRACK)
n_empty    = sum(1 for v in results.values() if len(v) == 0)
print(f"\nResults  ✓ {n_success:,} full  ⚠ {n_partial:,} partial  ✗ {n_empty:,} empty")

Fetching recommendations:   0%|          | 0/5000 [00:00<?, ?track/s]

WARNING │ 400 bad request for seed '1Qn7Jjvz4ndUyyAhbHRSHn': {"timestamp":"2026-03-17T07:35:26.053+00:00","error":"Cannot find any track for given seed ids, seeds need at least one 
WARNING │ 400 bad request for seed '5ls62WNKHUUrdF3r1cv83T': {"timestamp":"2026-03-17T07:35:26.267+00:00","error":"Cannot find any track for given seed ids, seeds need at least one 
WARNING │ 400 bad request for seed '5A3fPy30SN2wuzrahpcxvV': {"timestamp":"2026-03-17T07:35:26.367+00:00","error":"Cannot find any track for given seed ids, seeds need at least one 
WARNING │ 400 bad request for seed '6V91Cctseyb8yz67YQMDw3': {"timestamp":"2026-03-17T07:35:27.974+00:00","error":"Cannot find any track for given seed ids, seeds need at least one 
WARNING │ 400 bad request for seed '7xyYsOvq5Ec3P4fr6mM9fD': {"timestamp":"2026-03-17T07:35:28.296+00:00","error":"Cannot find any track for given seed ids, seeds need at least one 
WARNING │ 400 bad request for seed '6qItx3M2IZbXBKRnptbnHM': {"timestamp":"2026-03-17T07:3


Results  ✓ 2,601 full  ⚠ 1,805 partial  ✗ 594 empty


## 4 · Flatten to Polars DataFrame

In [8]:
rows = []
for seed_id, recs in results.items():
    for rank, rec in enumerate(recs, start=1):
        # Extract Spotify track ID from the href field
        rec_spotify_id = extract_spotify_id(rec.get("href"))

        # Artist names as pipe-separated string
        artists = rec.get("artists") or []
        artist_names = " | ".join(a.get("name", "") for a in artists if isinstance(a, dict))
        artist_ids   = " | ".join(
            (a.get("href", "").rstrip("/").split("/")[-1])
            for a in artists if isinstance(a, dict)
        )

        rows.append({
            # ── seed ──────────────────────────────────────────
            "seed_track_id":      seed_id,
            # ── recommendation ────────────────────────────────
            "rank":               rank,
            "rec_reccobeats_id":  rec.get("id"),
            "rec_track_id":       rec_spotify_id,          # Spotify track ID
            "rec_title":          rec.get("trackTitle"),
            "rec_artist_names":   artist_names,
            "rec_artist_ids":     artist_ids,
            "rec_duration_ms":    rec.get("durationMs"),
            "rec_popularity":     rec.get("popularity"),
            "rec_isrc":           rec.get("isrc"),
            "rec_spotify_href":   rec.get("href"),
        })

recs_df = pl.DataFrame(
    rows,
    schema={
        "seed_track_id":     pl.Utf8,
        "rank":              pl.Int32,
        "rec_reccobeats_id": pl.Utf8,
        "rec_track_id":      pl.Utf8,
        "rec_title":         pl.Utf8,
        "rec_artist_names":  pl.Utf8,
        "rec_artist_ids":    pl.Utf8,
        "rec_duration_ms":   pl.Int64,
        "rec_popularity":    pl.Int32,
        "rec_isrc":          pl.Utf8,
        "rec_spotify_href":  pl.Utf8,
    },
)

print(f"Recommendations DataFrame: {recs_df.shape[0]:,} rows × {recs_df.shape[1]} cols")
recs_df.head(5)

Recommendations DataFrame: 19,980 rows × 11 cols


seed_track_id,rank,rec_reccobeats_id,rec_track_id,rec_title,rec_artist_names,rec_artist_ids,rec_duration_ms,rec_popularity,rec_isrc,rec_spotify_href
str,i32,str,str,str,str,str,i64,i32,str,str
"""42tFTth2jcF7iSo0RBjfJF""",1,"""0ca46007-593d-47d5-a14e-a2b47b…","""41jXJkAfryMVJ1kRILtGSG""","""BOX""","""NCT DREAM""","""1gBUSTR3TyDdTVFIaQnc02""",170360,53,"""KRA302400039""","""https://open.spotify.com/track…"
"""42tFTth2jcF7iSo0RBjfJF""",2,"""5e976efc-21b9-4c80-940a-3559c0…","""7juaMHv1j8EAmEVEAlo1iT""","""Airhead""","""Honey Revenge""","""1DHMgO3IIYSYPJ6CFyDYnK""",163519,60,"""QM4TX2307126""","""https://open.spotify.com/track…"
"""42tFTth2jcF7iSo0RBjfJF""",3,"""6bd1c9d6-964b-4656-87b3-f3785e…","""7iGJ3ajScp5UusRzy4bNLg""","""UNA CAN""","""Sayf""","""3HAwumPgGOSXlZSyGWuLhB""",161610,58,"""ITQ002501429""","""https://open.spotify.com/track…"
"""42tFTth2jcF7iSo0RBjfJF""",4,"""ca2c2fac-9d62-47cb-a38f-73a207…","""7iInuFEObY92j9teGG6la1""","""Jhottya Barge Chore""","""Masoom Sharma | Ashu Twinkle |…","""36iDrP3UnCxsSH9LuSdkDj | 5J23f…",161525,56,"""UKJ8H2306949""","""https://open.spotify.com/track…"
"""4vVjPTApwXZwB2H3mBq3ml""",1,"""afa9ef09-65e5-4d36-a47b-3826d4…","""0M9ydKzuF3oZTfYYPfaGX1""","""Bad and Boujee (feat. Lil Uzi …","""Migos | Lil Uzi Vert""","""6oMuImdp5ZcFhWP0ESe6mG | 4O15N…",343150,69,"""QMCE31600796""","""https://open.spotify.com/track…"


## 5 · Join recommendations back to original DataFrame

In [ ]:
enriched_df = (
    df
    .join(
        recs_df,
        left_on  = TRACK_ID_COL,
        right_on = "seed_track_id",
        how      = "left",   # keep all original rows even if no recs returned
    )
)

print(f"Enriched DataFrame: {enriched_df.shape[0]:,} rows × {enriched_df.shape[1]} cols")
enriched_df.head(3)

Enriched DataFrame: 20,574 rows × 49 cols


track_id,original_name,original_artist,id,name,title,uri,type,duration,duration_ms,artists,preview_url,is_explicit,is_playable,album,release_date,metadata_status,lyrics,lyrics_status,stream_count,Unnamed: 0,artist_name,track_name,popularity,year,genre,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms_right,time_signature,rank,rec_reccobeats_id,rec_track_id,rec_title,rec_artist_names,rec_artist_ids,rec_duration_ms,rec_popularity,rec_isrc,rec_spotify_href
str,str,str,str,str,str,str,str,f64,f64,list[struct[3]],str,bool,bool,struct[3],str,str,str,str,f64,i64,str,str,i64,i64,str,f64,f64,i64,f64,i64,f64,f64,f64,f64,f64,f64,i64,i64,i32,str,str,str,str,str,i64,i32,str,str
"""0yLdNVWF3Srea0uzk55zFn""","""Flowers""","""Miley Cyrus""","""0yLdNVWF3Srea0uzk55zFn""","""Flowers""","""Flowers""","""spotify:track:0yLdNVWF3Srea0uz…","""track""",200454.0,200454.0,"[{""5YGY8feqx7naU7z4HrwZM6"",""Miley Cyrus"",""spotify:artist:5YGY8feqx7naU7z4HrwZM6""}]","""https://p.scdn.co/mp3-preview/…",false,true,"{[{300,""https://image-cdn-fa.spotifycdn.com/image/ab67616d00001e02f429549123dbe8552764ba1d"",300}, {64,""https://image-cdn-fa.spotifycdn.com/image/ab67616d00004851f429549123dbe8552764ba1d"",64}, {640,""https://image-cdn-fa.spotifycdn.com/image/ab67616d0000b273f429549123dbe8552764ba1d"",640}],"""",""album""}","""2023-01-13""","""ok""","""We were good, we were gold Kin…","""ok_lrclib""",2.8096e9,612503,"""Miley Cyrus""","""Flowers""",100,2023,"""pop""",0.707,0.681,0,-4.325,1,0.0668,0.0632,0.000005,0.0322,0.646,117.999,200455,4,null,null,null,null,null,null,null,null,null,null
"""4nrPB8O7Y7wsOCJdgXkthe""","""Shakira: Bzrp Music Sessions, …","""Bizarrap""","""4nrPB8O7Y7wsOCJdgXkthe""","""Shakira: Bzrp Music Sessions, …","""Shakira: Bzrp Music Sessions, …","""spotify:track:4nrPB8O7Y7wsOCJd…","""track""",214945.0,214945.0,"[{""716NhGYqD1jl2wI1Qkgq36"",""Bizarrap"",""spotify:artist:716NhGYqD1jl2wI1Qkgq36""}, {""0EmeFodog0BfCgMzAIvKQp"",""Shakira"",""spotify:artist:0EmeFodog0BfCgMzAIvKQp""}]","""https://p.scdn.co/mp3-preview/…",false,true,"{[{300,""https://image-cdn-fa.spotifycdn.com/image/ab67616d00001e02fdecba1ff443653b79ebb189"",300}, {64,""https://image-cdn-fa.spotifycdn.com/image/ab67616d00004851fdecba1ff443653b79ebb189"",64}, {640,""https://image-cdn-fa.spotifycdn.com/image/ab67616d0000b273fdecba1ff443653b79ebb189"",640}],"""",""album""}","""2023-01-11""","""ok""","""Pa tipos como tú-ú-ú-ú-ú Oh-oh…","""ok_lrclib""",1.1536e9,605178,"""Bizarrap""","""Shakira: Bzrp Music Sessions, …",96,2023,"""hip-hop""",0.778,0.632,2,-5.6,0,0.0493,0.274,0.0,0.0915,0.498,122.104,218289,4,1,"""213bd2d1-c5e8-4f6a-9e66-fe2a41…","""3xby7fOyqmeON8jsnom0AT""","""Nightcrawler (feat. Swae Lee &…","""Travis Scott | Swae Lee | Chie…","""0Y5tJX1MQlPlqiwlOH1tJY | 1zNqQ…",321560,77,"""USSM11506733""","""https://open.spotify.com/track…"
"""4nrPB8O7Y7wsOCJdgXkthe""","""Shakira: Bzrp Music Sessions, …","""Bizarrap""","""4nrPB8O7Y7wsOCJdgXkthe""","""Shakira: Bzrp Music Sessions, …","""Shakira: Bzrp Music Sessions, …","""spotify:track:4nrPB8O7Y7wsOCJd…","""track""",214945.0,214945.0,"[{""716NhGYqD1jl2wI1Qkgq36"",""Bizarrap"",""spotify:artist:716NhGYqD1jl2wI1Qkgq36""}, {""0EmeFodog0BfCgMzAIvKQp"",""Shakira"",""spotify:artist:0EmeFodog0BfCgMzAIvKQp""}]","""https://p.scdn.co/mp3-preview/…",false,true,"{[{300,""https://image-cdn-fa.spotifycdn.com/image/ab67616d00001e02fdecba1ff443653b79ebb189"",300}, {64,""https://image-cdn-fa.spotifycdn.com/image/ab67616d00004851fdecba1ff443653b79ebb189"",64}, {640,""https://image-cdn-fa.spotifycdn.com/image/ab67616d0000b273fdecba1ff443653b79ebb189"",640}],"""",""album""}","""2023-01-11""","""ok""","""Pa tipos como tú-ú-ú-ú-ú Oh-oh…","""ok_lrclib""",1.1536e9,605178,"""Bizarrap""","""Shakira: Bzrp Music Sessions, …",96,2023,"""hip-hop""",0.778,0.632,2,-5.6,0,0.0493,0.274,0.0,0.0915,0.498,122.104,218289,4,2,"""9e711c66-6e0d-43fe-b1ce-6ef4e0…","""3bnVBN67NBEzedqQuWrpP4""","""Tear in My Heart""",

## 6 · Validation

In [10]:
# Tracks with fewer than RECS_PER_TRACK recommendations
coverage = (
    enriched_df
    .group_by(TRACK_ID_COL)
    .agg(pl.col("rec_track_id").drop_nulls().len().alias("n_recs"))
    .with_columns(
        (pl.col("n_recs") >= RECS_PER_TRACK).alias("full_coverage")
    )
)

full  = coverage.filter(pl.col("full_coverage")).shape[0]
under = coverage.filter(~pl.col("full_coverage")).shape[0]
print(f"Full coverage (≥{RECS_PER_TRACK} recs): {full:,} tracks")
print(f"Under coverage:                        {under:,} tracks")

coverage.sort("n_recs").head(10)

Full coverage (≥5 recs): 2,601 tracks
Under coverage:                        2,399 tracks


track_id,n_recs,full_coverage
str,u32,bool
"""0N9C80kcgL0xXGduKnYKWi""",0,false
"""25ypHCQpDX2nrDFly7eZLZ""",0,false
"""1fPCGREltHJuV3axEPydLp""",0,false
"""1BZG99C7Co1r6QUC3zaS59""",0,false
"""2DPqR0kuZVjrOF5oxIRYPy""",0,false
"""43yyvWNjl3yWpghcrxZYkr""",0,false
"""7xNCacksfUkYXsXuSW4vNF""",0,false
"""1wLNEMiUzwvRZz9XHCXhAE""",0,false
"""0pt0wjZNeFOMIeCudmXRrl""",0,false


In [11]:
# Distribution of recommendation counts
(
    coverage
    .group_by("n_recs")
    .len()
    .sort("n_recs")
    .rename({"len": "tracks"})
)

n_recs,tracks
u32,u32
0,594
2,6
3,233
4,1566
5,2601


## 7 · Export

In [12]:
enriched_df.write_parquet(OUTPUT_PARQUET, compression="zstd")

size_mb = OUTPUT_PARQUET.stat().st_size / 1_048_576
print(f"✓ Saved → {OUTPUT_PARQUET}  ({size_mb:.2f} MB)")
print(f"  Shape : {enriched_df.shape[0]:,} rows × {enriched_df.shape[1]} cols")

✓ Saved → spotify_5000_reccobeats.parquet  (5.87 MB)
  Shape : 20,574 rows × 49 cols


## 8 · Quick peek at output

In [13]:
out = pl.read_parquet(OUTPUT_PARQUET)
print(out.schema)
out.head(5)

Schema({'track_id': String, 'original_name': String, 'original_artist': String, 'id': String, 'name': String, 'title': String, 'uri': String, 'type': String, 'duration': Float64, 'duration_ms': Float64, 'artists': List(Struct({'id': String, 'name': String, 'uri': String})), 'preview_url': String, 'is_explicit': Boolean, 'is_playable': Boolean, 'album': Struct({'images': List(Struct({'height': Int64, 'url': String, 'width': Int64})), 'name': String, 'type': String}), 'release_date': String, 'metadata_status': String, 'lyrics': String, 'lyrics_status': String, 'stream_count': Float64, 'Unnamed: 0': Int64, 'artist_name': String, 'track_name': String, 'popularity': Int64, 'year': Int64, 'genre': String, 'danceability': Float64, 'energy': Float64, 'key': Int64, 'loudness': Float64, 'mode': Int64, 'speechiness': Float64, 'acousticness': Float64, 'instrumentalness': Float64, 'liveness': Float64, 'valence': Float64, 'tempo': Float64, 'duration_ms_right': Int64, 'time_signature': Int64, 'rank':

track_id,original_name,original_artist,id,name,title,uri,type,duration,duration_ms,artists,preview_url,is_explicit,is_playable,album,release_date,metadata_status,lyrics,lyrics_status,stream_count,Unnamed: 0,artist_name,track_name,popularity,year,genre,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms_right,time_signature,rank,rec_reccobeats_id,rec_track_id,rec_title,rec_artist_names,rec_artist_ids,rec_duration_ms,rec_popularity,rec_isrc,rec_spotify_href
str,str,str,str,str,str,str,str,f64,f64,list[struct[3]],str,bool,bool,struct[3],str,str,str,str,f64,i64,str,str,i64,i64,str,f64,f64,i64,f64,i64,f64,f64,f64,f64,f64,f64,i64,i64,i32,str,str,str,str,str,i64,i32,str,str
"""0yLdNVWF3Srea0uzk55zFn""","""Flowers""","""Miley Cyrus""","""0yLdNVWF3Srea0uzk55zFn""","""Flowers""","""Flowers""","""spotify:track:0yLdNVWF3Srea0uz…","""track""",200454.0,200454.0,"[{""5YGY8feqx7naU7z4HrwZM6"",""Miley Cyrus"",""spotify:artist:5YGY8feqx7naU7z4HrwZM6""}]","""https://p.scdn.co/mp3-preview/…",false,true,"{[{300,""https://image-cdn-fa.spotifycdn.com/image/ab67616d00001e02f429549123dbe8552764ba1d"",300}, {64,""https://image-cdn-fa.spotifycdn.com/image/ab67616d00004851f429549123dbe8552764ba1d"",64}, {640,""https://image-cdn-fa.spotifycdn.com/image/ab67616d0000b273f429549123dbe8552764ba1d"",640}],"""",""album""}","""2023-01-13""","""ok""","""We were good, we were gold Kin…","""ok_lrclib""",2.8096e9,612503,"""Miley Cyrus""","""Flowers""",100,2023,"""pop""",0.707,0.681,0,-4.325,1,0.0668,0.0632,0.000005,0.0322,0.646,117.999,200455,4,null,null,null,null,null,null,null,null,null,null
"""4nrPB8O7Y7wsOCJdgXkthe""","""Shakira: Bzrp Music Sessions, …","""Bizarrap""","""4nrPB8O7Y7wsOCJdgXkthe""","""Shakira: Bzrp Music Sessions, …","""Shakira: Bzrp Music Sessions, …","""spotify:track:4nrPB8O7Y7wsOCJd…","""track""",214945.0,214945.0,"[{""716NhGYqD1jl2wI1Qkgq36"",""Bizarrap"",""spotify:artist:716NhGYqD1jl2wI1Qkgq36""}, {""0EmeFodog0BfCgMzAIvKQp"",""Shakira"",""spotify:artist:0EmeFodog0BfCgMzAIvKQp""}]","""https://p.scdn.co/mp3-preview/…",false,true,"{[{300,""https://image-cdn-fa.spotifycdn.com/image/ab67616d00001e02fdecba1ff443653b79ebb189"",300}, {64,""https://image-cdn-fa.spotifycdn.com/image/ab67616d00004851fdecba1ff443653b79ebb189"",64}, {640,""https://image-cdn-fa.spotifycdn.com/image/ab67616d0000b273fdecba1ff443653b79ebb189"",640}],"""",""album""}","""2023-01-11""","""ok""","""Pa tipos como tú-ú-ú-ú-ú Oh-oh…","""ok_lrclib""",1.1536e9,605178,"""Bizarrap""","""Shakira: Bzrp Music Sessions, …",96,2023,"""hip-hop""",0.778,0.632,2,-5.6,0,0.0493,0.274,0.0,0.0915,0.498,122.104,218289,4,1,"""213bd2d1-c5e8-4f6a-9e66-fe2a41…","""3xby7fOyqmeON8jsnom0AT""","""Nightcrawler (feat. Swae Lee &…","""Travis Scott | Swae Lee | Chie…","""0Y5tJX1MQlPlqiwlOH1tJY | 1zNqQ…",321560,77,"""USSM11506733""","""https://open.spotify.com/track…"
"""4nrPB8O7Y7wsOCJdgXkthe""","""Shakira: Bzrp Music Sessions, …","""Bizarrap""","""4nrPB8O7Y7wsOCJdgXkthe""","""Shakira: Bzrp Music Sessions, …","""Shakira: Bzrp Music Sessions, …","""spotify:track:4nrPB8O7Y7wsOCJd…","""track""",214945.0,214945.0,"[{""716NhGYqD1jl2wI1Qkgq36"",""Bizarrap"",""spotify:artist:716NhGYqD1jl2wI1Qkgq36""}, {""0EmeFodog0BfCgMzAIvKQp"",""Shakira"",""spotify:artist:0EmeFodog0BfCgMzAIvKQp""}]","""https://p.scdn.co/mp3-preview/…",false,true,"{[{300,""https://image-cdn-fa.spotifycdn.com/image/ab67616d00001e02fdecba1ff443653b79ebb189"",300}, {64,""https://image-cdn-fa.spotifycdn.com/image/ab67616d00004851fdecba1ff443653b79ebb189"",64}, {640,""https://image-cdn-fa.spotifycdn.com/image/ab67616d0000b273fdecba1ff443653b79ebb189"",640}],"""",""album""}","""2023-01-11""","""ok""","""Pa tipos como tú-ú-ú-ú-ú Oh-oh…","""ok_lrclib""",1.1536e9,605178,"""Bizarrap""","""Shakira: Bzrp Music Sessions, …",96,2023,"""hip-hop""",0.778,0.632,2,-5.6,0,0.0493,0.274,0.0,0.0915,0.498,122.104,218289,4,2,"""9e711c66-6e0d-43fe-b1ce-6ef4e0…","""3bnVBN67NBEzedqQuWrpP4""","""Tear in My Heart""",

In [14]:
# Sample: all recommendations for the first seed track
first_seed = track_ids[0]
out.filter(pl.col(TRACK_ID_COL) == first_seed).select(
    [TRACK_ID_COL, "rank", "rec_track_id", "rec_title", "rec_artist_names", "rec_popularity"]
)

track_id,rank,rec_track_id,rec_title,rec_artist_names,rec_popularity
str,i32,str,str,str,i32
"""42tFTth2jcF7iSo0RBjfJF""",1,"""41jXJkAfryMVJ1kRILtGSG""","""BOX""","""NCT DREAM""",53
"""42tFTth2jcF7iSo0RBjfJF""",2,"""7juaMHv1j8EAmEVEAlo1iT""","""Airhead""","""Honey Revenge""",60
"""42tFTth2jcF7iSo0RBjfJF""",3,"""7iGJ3ajScp5UusRzy4bNLg""","""UNA CAN""","""Sayf""",58
"""42tFTth2jcF7iSo0RBjfJF""",4,"""7iInuFEObY92j9teGG6la1""","""Jhottya Barge Chore""","""Masoom Sharma | Ashu Twinkle |…",56
